# Distil-ShortVU: Knowledge Distillation for Video Attractiveness Prediction

**Goal:** Predict how engaging a short social media video is (ECR score 0-1) and explain why.

**Approach:** Knowledge Distillation
- **Teacher models:** ImageBind (visual), pyiqa (quality), BLIP (caption), MiniLM (text encoding)
- **Student model:** Lightweight DistilStudent (~2.8M params) with gated multimodal fusion
- **KD Loss:** `L = L_ECR + 0.3*L_aesthetic + 0.3*L_technical + 0.3*L_KD_cosine`
- **Output:** Engagement score (ECR) + human-readable explanation

**Pipeline:** Extract Features -> Train Student -> Evaluate -> Predict with Explanations

## 1. Install Dependencies & Import Libraries

In [1]:
import os

# 1. Cài đặt các thư viện phụ trợ
!pip install pyiqa sentence-transformers eva-decord pytorchvideo timm

# 2. Clone ImageBind
bind_path = '/kaggle/working/ImageBind' if os.path.exists('/kaggle/working') else '/content/ImageBind'
if not os.path.exists(bind_path):
    !git clone https://github.com/facebookresearch/ImageBind.git {bind_path}

# 3. MA THUẬT: Cấm ImageBind phá hỏng các thư viện cốt lõi của hệ thống
!sed -i '/torchvision/d' {bind_path}/requirements.txt
!sed -i '/torchaudio/d' {bind_path}/requirements.txt
!sed -i '/transformers/d' {bind_path}/requirements.txt

# 4. Cài đặt ImageBind
!cd {bind_path} && pip install -e .

# 5. Cập nhật lại transformers lên bản mới nhất để tương thích với sentence-transformers
!pip install --upgrade transformers

print("All dependencies installed safely!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00


## 2. Configure Kaggle Paths & Constants

In [ ]:
import os
import sys
import json
import time
import gc
import warnings
import types
warnings.filterwarnings('ignore')

import torch
import torchvision
import torchvision.transforms.functional as F_t
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.stats import spearmanr

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime
from scipy import stats
from sklearn.model_selection import train_test_split

sys.modules['torchvision.transforms.functional_tensor'] = F_t
# 1. Tạo một module giả lập
dummy_module = types.ModuleType('torchvision.transforms.functional_tensor')

# 2. Copy toàn bộ các hàm từ functional sang module giả lập này
for attr in dir(F_t):
    setattr(dummy_module, attr, getattr(F_t, attr))

# 3. Bổ sung hàm _max_value mà pytorchvideo đang tìm kiếm
if not hasattr(dummy_module, '_max_value'):
    def _max_value(dtype):
        if dtype == torch.uint8: return 255
        elif dtype == torch.int8: return 127
        elif dtype == torch.int16: return 32767
        elif dtype == torch.int32: return 2147483647
        elif dtype == torch.int64: return 9223372036854775807
        else: return 1.0
    dummy_module._max_value = _max_value

# 4. Đăng ký module giả lập này vào hệ thống
sys.modules['torchvision.transforms.functional_tensor'] = dummy_module

# Add ImageBind to path
sys.path.insert(0, '/kaggle/working/ImageBind')

# ========================
# PATHS
# ========================
INPUT_DIR = "/kaggle/input/datasets/nguyntuncng/snapugc-dataset"
OUTPUT_DIR = "/kaggle/working"

TRAIN_CSV = os.path.join(INPUT_DIR, "train_data.csv")
VAL_CSV = os.path.join(INPUT_DIR, "val_data.csv")
TRAIN_VIDEO_DIR = os.path.join(INPUT_DIR, "train_videos/train_videos")
VAL_VIDEO_DIR = os.path.join(INPUT_DIR, "val_videos/val_videos")

TRAIN_FEATURES = os.path.join(OUTPUT_DIR, "train_features.json")
VAL_FEATURES = os.path.join(OUTPUT_DIR, "val_features.json")
TRAIN_SPLIT = os.path.join(OUTPUT_DIR, "train_split.json")
VAL_SPLIT = os.path.join(OUTPUT_DIR, "val_split.json")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, "predictions.json")
VAL_PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, "val_predictions.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ========================
# HYPERPARAMETERS
# ========================
HIDDEN_DIM = 512
VISUAL_DIM = 1024   # ImageBind embedding dimension
TEXT_DIM = 384       # MiniLM embedding dimension
DROPOUT = 0.2
BATCH_SIZE = 256
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
ECR_WEIGHT = 1.0
AES_WEIGHT = 0.3
TECH_WEIGHT = 0.3
KD_WEIGHT = 0.3
SAVE_EVERY = 5
NUM_SCORE_FRAMES = 3
NUM_VISUAL_FRAMES = 4
SEED = 42
SPLIT_RATIO = 0.9

# ========================
# DEVICE
# ========================
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

CPU_DEVICE = "cpu" if str(DEVICE) == "mps" else str(DEVICE)

print(f"Device: {DEVICE}")
print(f"Input dir: {INPUT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Hyperparams: batch={BATCH_SIZE}, lr={LEARNING_RATE}, epochs={EPOCHS}")
print(f"Loss weights: ECR={ECR_WEIGHT}, Aesthetic={AES_WEIGHT}, Technical={TECH_WEIGHT}, KD={KD_WEIGHT}")

Device: cuda
Input dir: /kaggle/input/datasets/nguyntuncng/snapugc-dataset
Output dir: /kaggle/working
Hyperparams: batch=256, lr=0.0003, epochs=20
Loss weights: ECR=1.0, Aesthetic=0.3, Technical=0.3, KD=0.3


## 3. Utility Functions

In [3]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total': total, 'trainable': trainable}

def save_json(data, path):
    os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f)

print("Utility functions defined")

Utility functions defined


## 4. Load & Explore CSV Data

In [4]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print(f"Training data: {train_df.shape}")
print(f"Validation data: {val_df.shape}")
print(f"\nTrain columns: {list(train_df.columns)}")
print(f"Val columns: {list(val_df.columns)}")

has_train_ecr = 'ECR' in train_df.columns
has_val_ecr = 'ECR' in val_df.columns
print(f"\nTrain has ECR: {has_train_ecr}")
print(f"Val has ECR: {has_val_ecr}")

if has_train_ecr:
    print(f"\n=== ECR Distribution ===")
    print(train_df['ECR'].describe())

print(f"\nMissing values (train):")
print(train_df.isnull().sum())

train_videos_list = [f for f in os.listdir(TRAIN_VIDEO_DIR) if f.endswith('.mp4')] if os.path.exists(TRAIN_VIDEO_DIR) else []
val_videos_list = [f for f in os.listdir(VAL_VIDEO_DIR) if f.endswith('.mp4')] if os.path.exists(VAL_VIDEO_DIR) else []
print(f"\nTrain videos found: {len(train_videos_list)}")
print(f"Val videos found: {len(val_videos_list)}")

train_df.head()

Training data: (106192, 5)
Validation data: (6000, 4)

Train columns: ['Id', 'Title', 'Description', 'Download_link', 'ECR']
Val columns: ['Id', 'Title', 'Description', 'Download_link']

Train has ECR: True
Val has ECR: False

=== ECR Distribution ===
count    106192.000000
mean          0.497539
std           0.290418
min           0.000000
25%           0.245766
50%           0.497081
75%           0.748484
max           0.999992
Name: ECR, dtype: float64

Missing values (train):
Id                   0
Title            66448
Description      60914
Download_link        0
ECR                  0
dtype: int64

Train videos found: 105015
Val videos found: 5937


,Id,Title,Description,Download_link,ECR
0,5902367b4d7d4c38fe4638593ddea7ee,zanzibarisland zanzibarbeach zanzibarlifest...,NaN,https://cf-st.sc-cdn.net/d/Pi2xtXMNjx6GfVEJnmo...,0.0
1,9be2d4d5e8d4bfd1f5ce59f51d7d96dd,NaN,NaN,https://cf-st.sc-cdn.net/d/p8788TQnO5KF94SH8Uz...,0.0
2,1e9b26ba367a8ed559916599294bbfc1,NaN,skullskullskullskullHAHAHA,https://cf-st.sc-cdn.net/d/7xbgKKhZWSVFpOnIAmN...,0.0
3,20c0ea87f1d3d7ce1f5480a8d748e48e,NaN,NaN,https://bolt-gcdn.sc-cdn.net/y/OWVh8dymJFW6fVo...,0.0
4,bf58c43be3a6c500eae6cb599b0f8935,NaN,NaN,https://cf-st.sc-cdn.net/h/Nwt7Zfg7rE31iGoJmMV...,0.0


## 5. Feature Extraction Pipeline

Extract 3 types of features per video:
1. **Visual embedding** (ImageBind, 1024-dim) - multimodal visual representation
2. **Quality scores** (pyiqa MUSIQ+TOPIQ) - aesthetic & technical quality (0-10)
3. **Text embedding** (all-MiniLM-L6-v2, 384-dim) - from title + description + BLIP caption

In [5]:
class QualityScorer:
    """pyiqa MUSIQ + TOPIQ scorer for aesthetic & technical quality."""

    def __init__(self):
        import pyiqa
        self.device = torch.device(CPU_DEVICE)
        self.musiq = pyiqa.create_metric('musiq', device=self.device)
        self.topiq = pyiqa.create_metric('topiq_nr', device=self.device)
        print(f"[QualityScorer] Loaded on {self.device}")

    def score(self, video_path, num_frames=3):
        """Score video quality from sampled frames."""
        try:
            from decord import VideoReader, cpu
            vr = VideoReader(video_path, ctx=cpu(0))
            total = len(vr)
            if total == 0:
                return {"aesthetic": 0.0, "technical": 0.0}
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
            frames = vr.get_batch(indices)
            frames = frames.asnumpy() if hasattr(frames, 'asnumpy') else frames.numpy()

            aes_scores, tech_scores = [], []
            for f in frames:
                t = torch.from_numpy(f).permute(2, 0, 1).float().unsqueeze(0) / 255.0
                t = t.to(self.device)
                with torch.no_grad():
                    aes_scores.append(self.musiq(t).item())
                    tech_scores.append(self.topiq(t).item())

            # Normalize: MUSIQ 0-100 -> 0-10, TOPIQ 0-1 -> 0-10
            aes = np.mean(aes_scores) / 10.0
            tech = np.mean(tech_scores) * 10.0
            return {
                "aesthetic": round(max(0, min(10, aes)), 2),
                "technical": round(max(0, min(10, tech)), 2),
            }
        except Exception as e:
            print(f"[QualityScorer] Error {video_path}: {e}")
            return {"aesthetic": 5.0, "technical": 5.0}

    def unload(self):
        del self.musiq, self.topiq
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


class VideoCaptioner:
    """BLIP captioner - single center frame."""

    def __init__(self):
        from transformers import BlipProcessor, BlipForConditionalGeneration
        model_id = "Salesforce/blip-image-captioning-base"
        self.processor = BlipProcessor.from_pretrained(model_id)
        self.model = BlipForConditionalGeneration.from_pretrained(model_id)
        self.dev = str(DEVICE)
        self.model = self.model.to(self.dev).eval()
        print(f"[VideoCaptioner] Loaded on {self.dev}")

    def caption(self, video_path):
        """Generate caption from center frame."""
        try:
            from decord import VideoReader, cpu
            vr = VideoReader(video_path, ctx=cpu(0))
            total = len(vr)
            if total == 0:
                return ""
            frame = vr[total // 2].asnumpy()
            image = Image.fromarray(frame)
            inputs = self.processor(images=image, return_tensors="pt")
            inputs = {k: v.to(self.dev) for k, v in inputs.items()}
            with torch.no_grad():
                ids = self.model.generate(**inputs, max_new_tokens=40)
            return self.processor.decode(ids[0], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"[VideoCaptioner] Error {video_path}: {e}")
            return ""

    def unload(self):
        del self.model, self.processor
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


class VisualEncoder:
    """ImageBind wrapper - vision only, 1024-dim embedding."""

    def __init__(self):
        from imagebind.models import imagebind_model
        from imagebind.models.imagebind_model import ModalityType
        self.ModalityType = ModalityType
        self.model = imagebind_model.imagebind_huge(pretrained=True)
        self.device = str(DEVICE)
        self.model = self.model.to(self.device).eval()
        print(f"[VisualEncoder] Loaded on {self.device}")

    def embed(self, video_path, num_frames=4):
        """Get 1024-dim visual embedding from video frames."""
        try:
            from decord import VideoReader, cpu
            from torchvision import transforms
            vr = VideoReader(video_path, ctx=cpu(0))
            total = len(vr)
            if total == 0:
                return None
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
            frames = vr.get_batch(indices)
            frames = frames.asnumpy() if hasattr(frames, 'asnumpy') else frames.numpy()

            preprocess = transforms.Compose([
                transforms.Resize(224),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.48145466, 0.4578275, 0.40821073],
                    std=[0.26862954, 0.26130258, 0.27577711]),
            ])

            tensors = [preprocess(Image.fromarray(f)) for f in frames]
            video_tensor = torch.stack(tensors).unsqueeze(0).to(self.device)
            with torch.no_grad():
                embs = self.model({self.ModalityType.VISION: video_tensor})
            emb = embs[self.ModalityType.VISION].cpu().numpy().flatten()
            emb = emb / (np.linalg.norm(emb) + 1e-8)
            return emb.tolist()
        except Exception as e:
            print(f"[VisualEncoder] Error {video_path}: {e}")
            return None

    def unload(self):
        del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


class TextEncoder:
    """Sentence-transformers text encoder for title+description+caption."""

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)
        self.dim = self.model.get_sentence_embedding_dimension()
        print(f"[TextEncoder] Loaded {model_name} (dim={self.dim})")

    def encode(self, title="", description="", caption=""):
        """Encode text metadata into a single embedding vector."""
        parts = []
        if title and str(title).strip():
            parts.append(f"Title: {str(title).strip()}")
        if description and str(description).strip():
            parts.append(f"Description: {str(description).strip()}")
        if caption and str(caption).strip():
            parts.append(f"Visual: {str(caption).strip()}")
        if not parts:
            return [0.0] * self.dim
        text = " | ".join(parts)
        emb = self.model.encode(text, normalize_embeddings=True)
        return emb.tolist()

    def encode_batch(self, texts):
        """Encode a batch of text strings."""
        embs = self.model.encode(texts, normalize_embeddings=True, batch_size=64)
        return embs.tolist()

    def unload(self):
        del self.model
        gc.collect()

print("Feature extractor classes defined")

Feature extractor classes defined


### 5.1 Run Feature Extraction

In [6]:
def extract_features(csv_file, video_folder, output_file, max_videos=None, save_every=500):
    """Extract features from videos for student model training."""
    print(f"\n{'='*60}")
    print(f"Feature Extraction | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Device: {DEVICE}")
    print(f"{'='*60}")

    df = pd.read_csv(csv_file)
    has_ecr = 'ECR' in df.columns
    print(f"CSV entries: {len(df)} | Has ECR: {has_ecr}")

    # Resume support
    existing = {}
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for item in json.load(f):
                vid_id = item.get('video_id', '')
                if vid_id and item.get('visual_emb') is not None:
                    existing[vid_id] = item
        print(f"Resuming: {len(existing)} already processed")

    work = []
    for _, row in df.iterrows():
        vid_id = str(row['Id'])
        vid_path = os.path.join(video_folder, f"{vid_id}.mp4")
        if vid_id not in existing and os.path.exists(vid_path):
            work.append({
                'id': vid_id, 'path': vid_path,
                'ecr': float(row['ECR']) if has_ecr and pd.notna(row.get('ECR')) else None,
                'title': str(row.get('Title', '')) if pd.notna(row.get('Title', '')) else '',
                'description': str(row.get('Description', '')) if pd.notna(row.get('Description', '')) else '',
            })

    if max_videos:
        work = work[:max_videos]

    print(f"To process: {len(work)} videos")
    if not work and not existing:
        print("Nothing to do!")
        return []

    est_hours = len(work) * 4.0 / 3600
    print(f"Estimated time: {est_hours:.1f} hours")

    print("\nLoading models...")
    scorer = QualityScorer()
    captioner = VideoCaptioner()
    visual_encoder = VisualEncoder()
    text_encoder = TextEncoder()

    results = list(existing.values())
    new_count = 0
    errors = 0
    t_start = time.time()

    pbar = tqdm(work, desc="Extracting features", unit="video")
    for item in pbar:
        try:
            quality = scorer.score(item['path'], NUM_SCORE_FRAMES)
            caption = captioner.caption(item['path'])
            visual_emb = visual_encoder.embed(item['path'], num_frames=NUM_VISUAL_FRAMES)
            text_emb = text_encoder.encode(
                title=item['title'], description=item['description'], caption=caption,
            )
            result = {
                'video_id': item['id'], 'video_path': item['path'],
                'ecr': item['ecr'], 'title': item['title'],
                'description': item['description'], 'caption': caption,
                'visual_emb': visual_emb, 'text_emb': text_emb,
                'quality_scores': quality,
            }
            results.append(result)
            new_count += 1

            elapsed = time.time() - t_start
            speed = new_count / elapsed if elapsed > 0 else 0
            remaining = (len(work) - new_count) / speed if speed > 0 else 0
            pbar.set_postfix({'new': new_count, 'spd': f'{speed:.1f}v/s', 'eta': f'{remaining/3600:.1f}h'})

            if new_count % save_every == 0:
                save_json(results, output_file)
                tqdm.write(f"  Saved {len(results)} total ({new_count} new)")
        except Exception as e:
            errors += 1
            tqdm.write(f"  Error on {item['id']}: {e}")

    save_json(results, output_file)

    scorer.unload(); captioner.unload()
    visual_encoder.unload(); text_encoder.unload()

    elapsed = time.time() - t_start
    print(f"\nDone! {new_count} new videos in {elapsed/3600:.1f}h | Total: {len(results)} | Errors: {errors}")
    if new_count > 0:
        print(f"Speed: {new_count/elapsed:.2f} v/s ({elapsed/new_count:.1f}s/video)")
    print(f"Output: {output_file}")
    return results

print("extract_features() defined")

extract_features() defined


In [7]:
# WARNING: This takes a long time (~4s/video)!
# 105k train ~ 116h, 6k val ~ 6.6h
# Set max_videos=100 for a quick test

print("=== Extracting TRAIN features ===")
train_results = extract_features(
    csv_file=TRAIN_CSV,
    video_folder=TRAIN_VIDEO_DIR,
    output_file=TRAIN_FEATURES,
    max_videos=5000,  # e.g. 100 for quick test
    save_every=500,
)

print("\n=== Extracting VAL features ===")
val_results = extract_features(
    csv_file=VAL_CSV,
    video_folder=VAL_VIDEO_DIR,
    output_file=VAL_FEATURES,
    max_videos=None,
    save_every=500,
)

print(f"\nFeature extraction complete!")

=== Extracting TRAIN features ===

Feature Extraction | 2026-04-01 13:15:34
Device: cuda
CSV entries: 106192 | Has ECR: True
To process: 5000 videos
Estimated time: 5.6 hours

Loading models...
Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/musiq_koniq_ckpt-e95806b9.pth" to /root/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth



100%|██████████| 104M/104M [00:00<00:00, 278MB/s] 


Loading pretrained model MUSIQ from /root/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Downloading: "https://huggingface.co/chaofengc/IQA-PyTorch-Weights/resolve/main/cfanet_nr_koniq_res50-9a73138b.pth" to /root/.cache/torch/hub/pyiqa/cfanet_nr_koniq_res50-9a73138b.pth



...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[TextEncoder] Loaded all-MiniLM-L6-v2 (dim=384)


Extracting features:   0%|          | 0/5937 [00:00<?, ?video/s]

  Saved 500 total (500 new)
  Saved 1000 total (1000 new)
  Saved 1500 total (1500 new)
  Saved 2000 total (2000 new)
  Saved 2500 total (2500 new)
  Saved 3000 total (3000 new)
  Saved 3500 total (3500 new)
  Saved 4000 total (4000 new)
  Saved 4500 total (4500 new)
  Saved 5000 total (5000 new)
  Saved 5500 total (5500 new)

Done! 5937 new videos in 2.6h | Total: 5937 | Errors: 0
Speed: 0.64 v/s (1.6s/video)
Output: /kaggle/working/val_features.json

Feature extraction complete!


### 5.2 Split Train Features into Train/Val

In [8]:
print("Loading train features for splitting...")
with open(TRAIN_FEATURES, 'r') as f:
    all_train_data = json.load(f)

valid_data = [d for d in all_train_data if d.get('visual_emb') is not None]
print(f"Total valid samples: {len(valid_data)}")

train_data, val_data = train_test_split(
    valid_data, test_size=1-SPLIT_RATIO, random_state=SEED
)
print(f"Train split: {len(train_data)} samples")
print(f"Val split: {len(val_data)} samples")

save_json(train_data, TRAIN_SPLIT)
save_json(val_data, VAL_SPLIT)
print(f"Saved: {TRAIN_SPLIT}")
print(f"Saved: {VAL_SPLIT}")

ecrs_train = [d['ecr'] for d in train_data if d.get('ecr') is not None]
ecrs_val = [d['ecr'] for d in val_data if d.get('ecr') is not None]
print(f"\nTrain ECR: mean={np.mean(ecrs_train):.4f}, std={np.std(ecrs_train):.4f}")
print(f"Val ECR:   mean={np.mean(ecrs_val):.4f}, std={np.std(ecrs_val):.4f}")

del all_train_data, valid_data
gc.collect()

Loading train features for splitting...
Total valid samples: 5000
Train split: 4500 samples
Val split: 500 samples
Saved: /kaggle/working/train_split.json
Saved: /kaggle/working/val_split.json

Train ECR: mean=0.0683, std=0.0534
Val ECR:   mean=0.0686, std=0.0531


83

## 6. Model Architecture: DistilStudent

Lightweight (~2.8M params) student model with **gated fusion** of visual and text embeddings.

Predicts: ECR (0-1, main task), Aesthetic quality (0-10), Technical quality (0-10).

Knowledge Distillation via **cosine embedding alignment** with teacher (ImageBind).

In [9]:
class DistilStudent(nn.Module):
    """Knowledge Distillation Student Model for video attractiveness prediction."""

    def __init__(self, visual_dim=1024, text_dim=384, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.visual_dim = visual_dim
        self.text_dim = text_dim
        self.hidden_dim = hidden_dim

        # Visual encoder
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # Text encoder
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # Gated cross-modal fusion
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # ECR prediction head (main task)
        self.ecr_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid(),
        )
        # Quality prediction heads (auxiliary / KD)
        self.aesthetic_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1),
        )
        self.technical_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1),
        )
        # KD projector: student hidden -> teacher embedding space
        self.kd_projector = nn.Sequential(
            nn.Linear(hidden_dim, visual_dim),
            nn.LayerNorm(visual_dim),
        )

    def forward(self, visual_emb, text_emb, ecr_targets=None,
                aesthetic_targets=None, technical_targets=None,
                teacher_emb=None, loss_weights=None):
        if loss_weights is None:
            loss_weights = {'ecr': 1.0, 'aesthetic': 0.3, 'technical': 0.3, 'kd': 0.5}

        v_hidden = self.visual_encoder(visual_emb)
        t_hidden = self.text_encoder(text_emb)

        concat = torch.cat([v_hidden, t_hidden], dim=-1)
        gate_weights = self.gate(concat)
        fused = self.fusion(concat)
        fused = gate_weights * fused

        predicted_ecr = self.ecr_head(fused).squeeze(-1)
        predicted_aesthetic = self.aesthetic_head(fused).squeeze(-1)
        predicted_technical = self.technical_head(fused).squeeze(-1)

        outputs = {
            'predicted_ecr': predicted_ecr,
            'predicted_aesthetic': predicted_aesthetic,
            'predicted_technical': predicted_technical,
            'visual_hidden': v_hidden,
            'text_hidden': t_hidden,
            'fused_hidden': fused,
            'gate_weights': gate_weights,
        }

        loss = torch.tensor(0.0, device=visual_emb.device)
        losses = {}

        if ecr_targets is not None:
            ecr_loss = F.mse_loss(predicted_ecr, ecr_targets)
            losses['ecr_loss'] = ecr_loss
            loss = loss + loss_weights['ecr'] * ecr_loss

        if aesthetic_targets is not None:
            aes_loss = F.mse_loss(predicted_aesthetic, aesthetic_targets)
            losses['aesthetic_loss'] = aes_loss
            loss = loss + loss_weights['aesthetic'] * aes_loss

        if technical_targets is not None:
            tech_loss = F.mse_loss(predicted_technical, technical_targets)
            losses['technical_loss'] = tech_loss
            loss = loss + loss_weights['technical'] * tech_loss

        if teacher_emb is not None:
            student_proj = self.kd_projector(fused)
            kd_loss = 1.0 - F.cosine_similarity(student_proj, teacher_emb, dim=-1).mean()
            losses['kd_loss'] = kd_loss
            loss = loss + loss_weights['kd'] * kd_loss

        outputs['loss'] = loss
        outputs['losses'] = losses
        return outputs

    def get_feature_importance(self, visual_emb, text_emb):
        """Compute feature importance via ablation (zero-out each modality)."""
        self.eval()
        with torch.no_grad():
            full_out = self.forward(visual_emb, text_emb)
            full_ecr = full_out['predicted_ecr'].item()

            vis_out = self.forward(visual_emb, torch.zeros_like(text_emb))
            vis_ecr = vis_out['predicted_ecr'].item()

            text_out = self.forward(torch.zeros_like(visual_emb), text_emb)
            text_ecr = text_out['predicted_ecr'].item()

            gate = full_out['gate_weights'].mean().item()

        return {
            'predicted_ecr': full_ecr,
            'visual_contribution': vis_ecr,
            'text_contribution': text_ecr,
            'visual_importance': abs(full_ecr - text_ecr),
            'text_importance': abs(full_ecr - vis_ecr),
            'gate_weight_mean': gate,
        }


def generate_explanation(predicted_ecr, quality_scores, feature_importance,
                         caption="", title=""):
    """Generate a human-readable explanation for the predicted engagement."""
    if predicted_ecr >= 0.8:
        engagement, emoji = "very high", "🔥"
    elif predicted_ecr >= 0.6:
        engagement, emoji = "high", "👍"
    elif predicted_ecr >= 0.4:
        engagement, emoji = "moderate", "➡️"
    elif predicted_ecr >= 0.2:
        engagement, emoji = "low", "👎"
    else:
        engagement, emoji = "very low", "⬇️"

    aes = quality_scores.get('aesthetic', 5.0)
    tech = quality_scores.get('technical', 5.0)
    aes_desc = "high visual appeal" if aes >= 7.0 else "moderate visual appeal" if aes >= 5.0 else "limited visual appeal"
    tech_desc = "excellent production quality" if tech >= 7.0 else "adequate production quality" if tech >= 5.0 else "basic production quality"

    vis_imp = feature_importance.get('visual_importance', 0)
    txt_imp = feature_importance.get('text_importance', 0)
    total_imp = vis_imp + txt_imp + 1e-8
    vis_pct = vis_imp / total_imp * 100
    txt_pct = txt_imp / total_imp * 100

    if vis_pct > 70:
        driver = "primarily driven by visual content"
    elif txt_pct > 70:
        driver = "primarily driven by title/description context"
    else:
        driver = "driven by both visual content and text context"

    parts = [
        f"{emoji} Predicted engagement: {engagement} (ECR={predicted_ecr:.2f}).",
        f"The video has {aes_desc} (aesthetic={aes:.1f}/10) and {tech_desc} (technical={tech:.1f}/10).",
        f"The prediction is {driver} (visual: {vis_pct:.0f}%, text: {txt_pct:.0f}%).",
    ]
    if caption:
        parts.append(f"Visual content: {caption}.")
    return " ".join(parts)


# Quick test
model_test = DistilStudent(hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
params_info = count_parameters(model_test)
print(f"DistilStudent parameters: {params_info['total']:,} ({params_info['trainable']:,} trainable)")

v = torch.randn(2, VISUAL_DIM).to(DEVICE)
t = torch.randn(2, TEXT_DIM).to(DEVICE)
out = model_test(v, t, ecr_targets=torch.rand(2).to(DEVICE))
print(f"Forward pass OK - ECR: {out['predicted_ecr'].detach().cpu().numpy()}")
print(f"Loss: {out['loss'].item():.4f}")
del model_test, v, t, out
gc.collect()
print("\nDistilStudent model defined and tested")

DistilStudent parameters: 2,828,803 (2,828,803 trainable)
Forward pass OK - ECR: [0.51752317 0.4801058 ]
Loss: 0.0991

DistilStudent model defined and tested


## 7. Dataset & DataLoader

In [10]:
class VideoFeaturesDataset(Dataset):
    """Dataset for extracted video features with KD targets."""

    def __init__(self, json_file, normalize_quality=True):
        print(f"Loading dataset from {json_file}...")
        with open(json_file, 'r') as f:
            raw_data = json.load(f)

        if isinstance(raw_data, dict):
            data_list = list(raw_data.values())
        else:
            data_list = raw_data

        self.data = []
        skipped = 0
        for item in data_list:
            if item.get('visual_emb') is not None:
                self.data.append(item)
            else:
                skipped += 1

        print(f"Loaded {len(self.data)} samples (skipped {skipped} without embeddings)")
        ecrs = [d['ecr'] for d in self.data if d.get('ecr') is not None]
        if ecrs:
            print(f"ECR stats: mean={np.mean(ecrs):.4f}, std={np.std(ecrs):.4f}, "
                  f"min={np.min(ecrs):.4f}, max={np.max(ecrs):.4f}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        visual_emb = torch.tensor(item['visual_emb'], dtype=torch.float32)
        text_emb = torch.tensor(item.get('text_emb', [0.0] * TEXT_DIM), dtype=torch.float32)
        ecr = item.get('ecr')
        ecr_tensor = torch.tensor(ecr if ecr is not None else 0.0, dtype=torch.float32)
        has_ecr = torch.tensor(1.0 if ecr is not None else 0.0, dtype=torch.float32)

        quality = item.get('quality_scores', item.get('aesthetic_score', {}))
        aesthetic = quality.get('aesthetic', 5.0) / 10.0
        technical = quality.get('technical', 5.0) / 10.0
        teacher_emb = visual_emb.clone()

        return {
            'visual_emb': visual_emb,
            'text_emb': text_emb,
            'ecr': ecr_tensor,
            'has_ecr': has_ecr,
            'aesthetic': torch.tensor(aesthetic, dtype=torch.float32),
            'technical': torch.tensor(technical, dtype=torch.float32),
            'teacher_emb': teacher_emb,
        }

print("VideoFeaturesDataset defined")

VideoFeaturesDataset defined


## 8. Training & Evaluation Functions

In [11]:
def train_epoch(model, dataloader, optimizer, device, epoch, loss_weights):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    loss_accum = {}
    num_batches = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}")
    for batch in pbar:
        visual_emb = batch['visual_emb'].to(device)
        text_emb = batch['text_emb'].to(device)
        ecr_targets = batch['ecr'].to(device)
        aesthetic_targets = batch['aesthetic'].to(device)
        technical_targets = batch['technical'].to(device)
        teacher_emb = batch['teacher_emb'].to(device)

        optimizer.zero_grad()
        outputs = model(
            visual_emb=visual_emb, text_emb=text_emb,
            ecr_targets=ecr_targets,
            aesthetic_targets=aesthetic_targets,
            technical_targets=technical_targets,
            teacher_emb=teacher_emb,
            loss_weights=loss_weights,
        )
        loss = outputs['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        for k, v in outputs['losses'].items():
            loss_accum[k] = loss_accum.get(k, 0) + v.item()
        num_batches += 1
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'ecr': f'{outputs["losses"].get("ecr_loss", torch.tensor(0)).item():.4f}',
        })

    metrics = {'loss': total_loss / num_batches}
    for k, v in loss_accum.items():
        metrics[k] = v / num_batches
    return metrics


def evaluate(model, dataloader, device, loss_weights):
    """Evaluate model with comprehensive metrics."""
    model.eval()
    total_loss = 0
    all_ecr_pred, all_ecr_true = [], []
    num_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            visual_emb = batch['visual_emb'].to(device)
            text_emb = batch['text_emb'].to(device)
            ecr_targets = batch['ecr'].to(device)
            has_ecr = batch['has_ecr'].to(device)
            aesthetic_targets = batch['aesthetic'].to(device)
            technical_targets = batch['technical'].to(device)
            teacher_emb = batch['teacher_emb'].to(device)

            outputs = model(
                visual_emb=visual_emb, text_emb=text_emb,
                ecr_targets=ecr_targets,
                aesthetic_targets=aesthetic_targets,
                technical_targets=technical_targets,
                teacher_emb=teacher_emb,
                loss_weights=loss_weights,
            )
            total_loss += outputs['loss'].item()

            mask = has_ecr.bool()
            if mask.any():
                all_ecr_pred.extend(outputs['predicted_ecr'][mask].cpu().numpy())
                all_ecr_true.extend(ecr_targets[mask].cpu().numpy())
            num_batches += 1

    metrics = {'loss': total_loss / num_batches}
    if all_ecr_pred:
        ecr_pred = np.array(all_ecr_pred)
        ecr_true = np.array(all_ecr_true)
        metrics['ecr_mse'] = np.mean((ecr_pred - ecr_true) ** 2)
        metrics['ecr_mae'] = np.mean(np.abs(ecr_pred - ecr_true))
        metrics['ecr_pearson'] = np.corrcoef(ecr_pred, ecr_true)[0, 1] if len(ecr_pred) > 1 else 0
        metrics['ecr_spearman'] = spearmanr(ecr_pred, ecr_true).correlation if len(ecr_pred) > 1 else 0
    return metrics

print("train_epoch() and evaluate() defined")

train_epoch() and evaluate() defined


## 9. Training Loop

Main training loop with:
- AdamW optimizer + CosineAnnealing LR scheduler
- Gradient clipping (max_norm=1.0)
- Best model saving by validation loss
- History tracking for visualization

In [12]:
# --- Create datasets ---
# Use train/val SPLIT files (which have ECR labels from train set)
train_dataset = VideoFeaturesDataset(TRAIN_SPLIT)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

val_dataset = VideoFeaturesDataset(VAL_SPLIT)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# --- Create model ---
model = DistilStudent(hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
params = count_parameters(model)
print(f"Model parameters: {params['total']:,} ({params['trainable']:,} trainable)")

# --- Optimizer & scheduler ---
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.01)

# --- Training ---
best_val_loss = float('inf')
best_ecr_corr = 0
history = {'train_loss': [], 'val_loss': [], 'ecr_pearson': [], 'ecr_spearman': [], 'ecr_mae': [], 'lr': []}

LOSS_WEIGHTS = {
    'ecr': ECR_WEIGHT,
    'aesthetic': AES_WEIGHT,
    'technical': TECH_WEIGHT,
    'kd': KD_WEIGHT
}

BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model.pth")
LAST_MODEL_PATH = os.path.join(OUTPUT_DIR, "last_model.pth")

print(f"\nStarting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}")
print(f"Loss weights: {LOSS_WEIGHTS}")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_epoch(model, train_loader, optimizer, DEVICE, epoch, LOSS_WEIGHTS)
    scheduler.step()
    val_metrics = evaluate(model, val_loader, DEVICE, LOSS_WEIGHTS)

    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['ecr_pearson'].append(val_metrics.get('ecr_pearson', 0))
    history['ecr_spearman'].append(val_metrics.get('ecr_spearman', 0))
    history['ecr_mae'].append(val_metrics.get('ecr_mae', 0))
    history['lr'].append(optimizer.param_groups[0]['lr'])

    log = f"Epoch {epoch}: train_loss={train_metrics['loss']:.4f}, val_loss={val_metrics['loss']:.4f}"
    if 'ecr_pearson' in val_metrics:
        log += f", ECR_r={val_metrics['ecr_pearson']:.4f}, ECR_rho={val_metrics['ecr_spearman']:.4f}, MAE={val_metrics['ecr_mae']:.4f}"
    print(log)

    # Save best
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_ecr_corr = val_metrics.get('ecr_pearson', 0)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'loss_weights': LOSS_WEIGHTS,
            'history': history,
            'args': {'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT},
        }, BEST_MODEL_PATH)
        print(f"  \u2605 New best model saved (loss={best_val_loss:.4f}, ECR_r={best_ecr_corr:.4f})")

    # Periodic save
    if epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'train_metrics': train_metrics,
            'val_metrics': val_metrics,
            'loss_weights': LOSS_WEIGHTS,
            'history': history,
            'args': {'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT},
        }, os.path.join(OUTPUT_DIR, f'distil_student_epoch{epoch}.pth'))

print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Best ECR Pearson: {best_ecr_corr:.4f}")
print(f"Best model: {BEST_MODEL_PATH}")

Loading dataset from /kaggle/working/train_split.json...
Loaded 4500 samples (skipped 0 without embeddings)
ECR stats: mean=0.0683, std=0.0534, min=0.0000, max=0.5216
Loading dataset from /kaggle/working/val_split.json...
Loaded 500 samples (skipped 0 without embeddings)
ECR stats: mean=0.0686, std=0.0531, min=0.0000, max=0.2493
Model parameters: 2,828,803 (2,828,803 trainable)

Starting training: 20 epochs, batch_size=256
Loss weights: {'ecr': 1.0, 'aesthetic': 0.3, 'technical': 0.3, 'kd': 0.3}


Epoch 1:   0%|          | 0/18 [00:00<?, ?it/s]

NameError: name 'spearmanr' is not defined

## 10. Evaluation & Prediction

Load the best checkpoint and run full evaluation on the validation set.

In [ ]:
def load_best_model(checkpoint_path, device):
    """Load the best trained model."""
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    args = checkpoint.get('args', {})
    model = DistilStudent(
        hidden_dim=args.get('hidden_dim', HIDDEN_DIM),
        dropout=args.get('dropout', DROPOUT),
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    epoch = checkpoint.get('epoch', 'unknown')
    print(f"Loaded from epoch {epoch}")
    if 'val_metrics' in checkpoint and checkpoint['val_metrics']:
        vm = checkpoint['val_metrics']
        print(f"  Val loss: {vm.get('loss', 0):.4f}")
        if 'ecr_pearson' in vm:
            print(f"  ECR Pearson: {vm['ecr_pearson']:.4f}, Spearman: {vm['ecr_spearman']:.4f}")
    return model


def predict_batch(model, data_path, device, explain=False):
    """Run predictions on a JSON dataset."""
    print(f"Loading data from {data_path}")
    with open(data_path, 'r') as f:
        data = json.load(f)

    if isinstance(data, dict):
        data_list = list(data.values())
    else:
        data_list = data

    results = []
    for item in tqdm(data_list, desc="Predicting"):
        if item.get('visual_emb') is None:
            continue

        v = torch.tensor(item['visual_emb'], dtype=torch.float32).unsqueeze(0).to(device)
        t = torch.tensor(item.get('text_emb', [0.0]*TEXT_DIM), dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(visual_emb=v, text_emb=t)

        pred_ecr = outputs['predicted_ecr'].item()
        pred_aes = outputs['predicted_aesthetic'].item() * 10
        pred_tech = outputs['predicted_technical'].item() * 10
        quality = item.get('quality_scores', item.get('aesthetic_score', {}))

        result = {
            'video_id': item.get('video_id', os.path.basename(item.get('video_path', ''))),
            'video_path': item.get('video_path', ''),
            'true_ecr': item.get('ecr'),
            'predicted_ecr': pred_ecr,
            'predicted_aesthetic': pred_aes,
            'predicted_technical': pred_tech,
            'true_aesthetic': quality.get('aesthetic', 0),
            'true_technical': quality.get('technical', 0),
        }

        if explain:
            importance = model.get_feature_importance(v, t)
            result['explanation'] = generate_explanation(
                predicted_ecr=pred_ecr,
                quality_scores=quality if quality else {'aesthetic': 5.0, 'technical': 5.0},
                feature_importance=importance,
                caption=item.get('caption', ''),
                title=item.get('title', ''),
            )
            result['feature_importance'] = importance

        results.append(result)

    return results

print("load_best_model() and predict_batch() defined")

### 10.1 Run evaluation on validation set

In [ ]:
# Load best model
best_model = load_best_model(BEST_MODEL_PATH, DEVICE)

# Run predictions with explanations on VAL_SPLIT (has ECR labels)
val_results = predict_batch(best_model, VAL_SPLIT, DEVICE, explain=True)

# Compute metrics
from scipy.stats import spearmanr, kendalltau

has_ecr = [r for r in val_results if r.get('true_ecr') is not None]
if has_ecr:
    true_ecr = np.array([r['true_ecr'] for r in has_ecr])
    pred_ecr = np.array([r['predicted_ecr'] for r in has_ecr])

    pearson_r = np.corrcoef(pred_ecr, true_ecr)[0, 1]
    spearman_r = spearmanr(pred_ecr, true_ecr).correlation
    kendall_r = kendalltau(pred_ecr, true_ecr).correlation
    mae = np.mean(np.abs(pred_ecr - true_ecr))
    mse = np.mean((pred_ecr - true_ecr)**2)

    print(f"\n{'='*50}")
    print("Evaluation Metrics")
    print(f"{'='*50}")
    print(f"Samples:       {len(has_ecr)}")
    print(f"ECR Pearson:   {pearson_r:.4f}")
    print(f"ECR Spearman:  {spearman_r:.4f}")
    print(f"ECR Kendall:   {kendall_r:.4f}")
    print(f"ECR MAE:       {mae:.4f}")
    print(f"ECR MSE:       {mse:.6f}")

    binary_true = (true_ecr > 0.5).astype(int)
    binary_pred = (pred_ecr > 0.5).astype(int)
    acc = np.mean(binary_true == binary_pred)
    print(f"Binary Acc:    {acc:.4f}")
else:
    print("WARNING: No samples with ECR labels found!")

# Save predictions
predictions_path = os.path.join(OUTPUT_DIR, 'val_predictions.json')
with open(predictions_path, 'w') as f:
    json.dump(val_results, f, indent=2, ensure_ascii=False)
print(f"\nPredictions saved to {predictions_path}")

# Show sample explanations
explained = [r for r in val_results if r.get('explanation')]
if explained:
    print(f"\n--- Sample Explanations ---")
    for r in explained[:5]:
        print(f"\n  Video: {r['video_id']}")
        if r['true_ecr'] is not None:
            print(f"  True ECR:  {r['true_ecr']:.4f}")
        else:
            print(f"  True ECR:  N/A")
        print(f"  Pred ECR:  {r['predicted_ecr']:.4f}")
        print(f"  {r['explanation']}")

## 11. Visualization

Training curves, error distributions, and prediction scatter plots.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Training & Validation loss
ax = axes[0, 0]
ax.plot(history['train_loss'], label='Train Loss', color='blue')
ax.plot(history['val_loss'], label='Val Loss', color='red')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# 2) ECR Correlation over epochs
ax = axes[0, 1]
ax.plot(history['ecr_pearson'], label='Pearson r', color='green')
ax.plot(history['ecr_spearman'], label='Spearman rho', color='orange')
ax.set_xlabel('Epoch')
ax.set_ylabel('Correlation')
ax.set_title('ECR Correlation Metrics')
ax.legend()
ax.grid(True, alpha=0.3)

# 3) Prediction scatter plot
ax = axes[1, 0]
if has_ecr:
    ax.scatter(true_ecr, pred_ecr, alpha=0.3, s=10, color='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Ideal')
    ax.set_xlabel('True ECR')
    ax.set_ylabel('Predicted ECR')
    ax.set_title(f'ECR Predictions (r={np.corrcoef(pred_ecr, true_ecr)[0,1]:.3f})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)

# 4) Error distribution
ax = axes[1, 1]
if has_ecr:
    errors = pred_ecr - true_ecr
    ax.hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Prediction Error (pred - true)')
    ax.set_ylabel('Count')
    ax.set_title(f'Error Distribution (MAE={np.mean(np.abs(errors)):.4f})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Training results plot saved")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# True vs Predicted ECR distribution
ax = axes[0]
if has_ecr:
    ax.hist(true_ecr, bins=50, alpha=0.6, label='True ECR', color='blue', edgecolor='white')
    ax.hist(pred_ecr, bins=50, alpha=0.6, label='Predicted ECR', color='orange', edgecolor='white')
    ax.set_xlabel('ECR')
    ax.set_ylabel('Count')
    ax.set_title('ECR Distribution: True vs Predicted')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Learning rate schedule
ax = axes[1]
ax.plot(history['lr'], color='purple')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule (CosineAnnealing)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 11.1 Feature Importance Analysis

Analyze how much each modality (visual vs text) contributes to predictions.

In [ ]:
# Analyze feature importance on a sample of validation data
print("Analyzing feature importance...")
n_samples = min(100, len(val_results))
vis_importance = []
txt_importance = []

with open(VAL_SPLIT, 'r') as f:
    val_data_raw = json.load(f)
if isinstance(val_data_raw, dict):
    val_data_list = list(val_data_raw.values())
else:
    val_data_list = val_data_raw

for item in tqdm(val_data_list[:n_samples], desc="Feature importance"):
    if item.get('visual_emb') is None:
        continue
    v = torch.tensor(item['visual_emb'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    t = torch.tensor(item.get('text_emb', [0.0]*TEXT_DIM), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    imp = best_model.get_feature_importance(v, t)
    vis_importance.append(imp['visual_importance'])
    txt_importance.append(imp['text_importance'])

vis_arr = np.array(vis_importance)
txt_arr = np.array(txt_importance)

fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Visual (ImageBind)', 'Text (Title+Desc+Caption)']
means = [vis_arr.mean(), txt_arr.mean()]
stds = [vis_arr.std(), txt_arr.std()]
bars = ax.bar(labels, means, yerr=stds, capsize=10, color=['#4C72B0', '#DD8452'], alpha=0.8)
ax.set_ylabel('Importance (ablation delta)')
ax.set_title(f'Feature Importance (n={n_samples})')
ax.grid(True, alpha=0.3, axis='y')

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

total = vis_arr.mean() + txt_arr.mean() + 1e-8
print(f"\nVisual contribution: {vis_arr.mean()/total*100:.1f}%")
print(f"Text contribution:   {txt_arr.mean()/total*100:.1f}%")

### 11.2 Inference Time Measurement

Measure single-sample inference latency for the student model.

In [ ]:
# Measure inference latency
import time

best_model.eval()
times = []
v_dummy = torch.randn(1, VISUAL_DIM).to(DEVICE)
t_dummy = torch.randn(1, TEXT_DIM).to(DEVICE)

# Warmup
for _ in range(20):
    with torch.no_grad():
        _ = best_model(visual_emb=v_dummy, text_emb=t_dummy)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()

# Measure
for _ in range(500):
    start = time.time()
    with torch.no_grad():
        _ = best_model(visual_emb=v_dummy, text_emb=t_dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    times.append(time.time() - start)

inference_ms = np.mean(times) * 1000
inference_std = np.std(times) * 1000
throughput = 1.0 / np.mean(times)

print(f"Student Model Inference Time:")
print(f"  Latency:    {inference_ms:.2f} \u00b1 {inference_std:.2f} ms/sample")
print(f"  Throughput: {throughput:.0f} samples/sec")
print(f"  Parameters: {count_parameters(best_model)['total']:,}")

### 11.3 Gate Weights Distribution

Visualize how the gated fusion mechanism weights visual vs text modalities.

In [ ]:
# Collect gate weights across validation set
gate_means = []

with open(VAL_SPLIT, 'r') as f:
    val_items = json.load(f)
if isinstance(val_items, dict):
    val_items = list(val_items.values())

for item in tqdm(val_items, desc="Collecting gate weights"):
    if item.get('visual_emb') is None:
        continue
    v = torch.tensor(item['visual_emb'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    t = torch.tensor(item.get('text_emb', [0.0]*TEXT_DIM), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = best_model(visual_emb=v, text_emb=t)
    gate_means.append(out['gate_weights'].cpu().numpy().flatten().mean())

gate_arr = np.array(gate_means)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(gate_arr, bins=50, color='teal', alpha=0.7, edgecolor='white')
ax.axvline(x=gate_arr.mean(), color='red', linestyle='--', label=f'Mean={gate_arr.mean():.3f}')
ax.set_xlabel('Mean Gate Weight')
ax.set_ylabel('Count')
ax.set_title('Gated Fusion: Gate Weight Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ecrs_for_gate = [item.get('ecr', 0) for item in val_items if item.get('visual_emb') is not None]
valid_ecrs = [(g, e) for g, e in zip(gate_means, ecrs_for_gate) if e is not None]
if valid_ecrs:
    g_vals, e_vals = zip(*valid_ecrs)
    ax.scatter(e_vals, g_vals, alpha=0.3, s=10, color='teal')
    ax.set_xlabel('ECR')
    ax.set_ylabel('Mean Gate Weight')
    ax.set_title('Gate Weight vs ECR')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gate_weights.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Gate weight stats: mean={gate_arr.mean():.4f}, std={gate_arr.std():.4f}")

### 11.4 Error Analysis by ECR Range

Break down prediction performance by ECR segment.

In [ ]:
# Error analysis by ECR segment
if has_ecr:
    ranges = [(0, 0.1), (0.1, 0.2), (0.2, 0.3), (0.3, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
    print(f"\n{'ECR Range':<12} {'Count':>6} {'MAE':>8} {'RMSE':>8} {'Pearson':>8}")
    print("-" * 50)

    range_data = []
    for lo, hi in ranges:
        mask = (true_ecr >= lo) & (true_ecr < hi)
        n = mask.sum()
        if n > 5:
            mae_r = np.mean(np.abs(pred_ecr[mask] - true_ecr[mask]))
            rmse_r = np.sqrt(np.mean((pred_ecr[mask] - true_ecr[mask])**2))
            r = np.corrcoef(pred_ecr[mask], true_ecr[mask])[0, 1] if n > 2 else float('nan')
            print(f"[{lo:.1f}-{hi:.1f})    {n:>6} {mae_r:>8.4f} {rmse_r:>8.4f} {r:>8.4f}")
            range_data.append({'range': f'{lo:.1f}-{hi:.1f}', 'count': int(n), 'mae': mae_r, 'rmse': rmse_r})
        else:
            print(f"[{lo:.1f}-{hi:.1f})    {n:>6}      N/A      N/A      N/A")

    if range_data:
        fig, ax = plt.subplots(figsize=(10, 5))
        x = [d['range'] for d in range_data]
        y_mae = [d['mae'] for d in range_data]
        y_count = [d['count'] for d in range_data]

        ax2 = ax.twinx()
        ax2.bar(x, y_count, alpha=0.2, color='steelblue', label='Count')
        ax.plot(x, y_mae, 'ro-', linewidth=2, markersize=8, label='MAE')
        ax.set_xlabel('ECR Range')
        ax.set_ylabel('MAE', color='red')
        ax2.set_ylabel('Sample Count', color='steelblue')
        ax.set_title('Prediction Error by ECR Range')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper left')
        ax2.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'error_by_range.png'), dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("No ECR labels available for error analysis")

## 13. Ablation Study

Systematic experiments to validate design choices:
1. **Modality comparison**: Visual-only vs Text-only vs Visual+Text
2. **Fusion method**: Concatenation vs Gated Fusion
3. **Training strategy**: ECR-only vs Multi-task vs Multi-task + KD

In [ ]:
# === Ablation helpers ===
from scipy.stats import spearmanr, kendalltau
import copy

def train_ablation_model(model, train_loader, val_loader, device,
                         loss_weights, epochs=50, lr=3e-4, name="model"):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)
    best_loss = float('inf')
    best_state = None
    hist = {'train_loss': [], 'val_loss': [], 'ecr_pearson': []}
    for epoch in range(1, epochs + 1):
        train_m = train_epoch(model, train_loader, optimizer, device, epoch, loss_weights)
        scheduler.step()
        val_m = evaluate(model, val_loader, device, loss_weights)
        hist['train_loss'].append(train_m['loss'])
        hist['val_loss'].append(val_m['loss'])
        hist['ecr_pearson'].append(val_m.get('ecr_pearson', 0))
        if val_m['loss'] < best_loss:
            best_loss = val_m['loss']
            best_state = copy.deepcopy(model.state_dict())
        if epoch % 10 == 0 or epoch == epochs:
            print(f"  [{name}] Epoch {epoch}: val_loss={val_m['loss']:.4f}, ECR_r={val_m.get('ecr_pearson', 0):.4f}")
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model, hist

def evaluate_full_metrics(model, data_path, device, mode='full'):
    with open(data_path, 'r') as f:
        data = json.load(f)
    data_list = data if isinstance(data, list) else list(data.values())
    preds, trues = [], []
    for item in data_list:
        if item.get('visual_emb') is None or item.get('ecr') is None:
            continue
        v = torch.tensor(item['visual_emb'], dtype=torch.float32).unsqueeze(0).to(device)
        t = torch.tensor(item.get('text_emb', [0.0]*TEXT_DIM), dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            if mode == 'visual_only':
                out = model(visual_emb=v, text_emb=torch.zeros_like(t))
            elif mode == 'text_only':
                out = model(visual_emb=torch.zeros_like(v), text_emb=t)
            else:
                out = model(visual_emb=v, text_emb=t)
        preds.append(out['predicted_ecr'].item())
        trues.append(item['ecr'])
    preds, trues = np.array(preds), np.array(trues)
    if len(preds) < 3:
        return {'pearson': 0, 'spearman': 0, 'kendall': 0, 'mae': 0, 'mse': 0, 'n': len(preds)}
    return {
        'pearson': float(np.corrcoef(preds, trues)[0, 1]),
        'spearman': float(spearmanr(preds, trues).correlation),
        'kendall': float(kendalltau(preds, trues).correlation),
        'mae': float(np.mean(np.abs(preds - trues))),
        'mse': float(np.mean((preds - trues)**2)),
        'n': len(preds),
    }

print("Ablation helpers defined")

### 13.1 Ablation: Modality Comparison

Evaluate visual-only, text-only, and full (visual+text) at inference time.

In [ ]:
print("=" * 60)
print("ABLATION 1: Modality Comparison")
print("=" * 60)

modality_results = {}
for mode in ['full', 'visual_only', 'text_only']:
    metrics = evaluate_full_metrics(best_model, VAL_SPLIT, DEVICE, mode)
    modality_results[mode] = metrics
    print(f"\n{mode:12s}: Pearson={metrics['pearson']:.4f}, "
          f"Spearman={metrics['spearman']:.4f}, "
          f"Kendall={metrics['kendall']:.4f}, "
          f"MAE={metrics['mae']:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
modes = ['Visual+Text\n(Full)', 'Visual Only', 'Text Only']
pearson_vals = [modality_results[m]['pearson'] for m in ['full', 'visual_only', 'text_only']]
spearman_vals = [modality_results[m]['spearman'] for m in ['full', 'visual_only', 'text_only']]
mae_vals = [modality_results[m]['mae'] for m in ['full', 'visual_only', 'text_only']]

x = np.arange(len(modes))
w = 0.25
bars1 = ax.bar(x - w, pearson_vals, w, label='Pearson', color='#4C72B0', alpha=0.8)
bars2 = ax.bar(x, spearman_vals, w, label='Spearman', color='#55A868', alpha=0.8)
bars3 = ax.bar(x + w, mae_vals, w, label='MAE', color='#DD8452', alpha=0.8)
ax.set_xlabel('Modality')
ax.set_ylabel('Score')
ax.set_title('Ablation 1: Modality Comparison')
ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.005, f'{h:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ablation_modality.png'), dpi=150, bbox_inches='tight')
plt.show()

### 13.2 Ablation: Fusion Method (Concatenation vs Gated)

Compare simple concatenation with gated fusion.

In [ ]:
class ConcatStudent(nn.Module):
    """Baseline: simple concatenation (no gating)."""
    def __init__(self, visual_dim=1024, text_dim=384, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.visual_dim = visual_dim
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.text_encoder = nn.Sequential(
            nn.Linear(text_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.ecr_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1), nn.Sigmoid())
        self.aesthetic_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4), nn.GELU(), nn.Linear(hidden_dim // 4, 1))
        self.technical_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4), nn.GELU(), nn.Linear(hidden_dim // 4, 1))
        self.kd_projector = nn.Sequential(
            nn.Linear(hidden_dim, visual_dim), nn.LayerNorm(visual_dim))

    def forward(self, visual_emb, text_emb, ecr_targets=None,
                aesthetic_targets=None, technical_targets=None,
                teacher_emb=None, loss_weights=None):
        if loss_weights is None:
            loss_weights = {'ecr': 1.0, 'aesthetic': 0.3, 'technical': 0.3, 'kd': 0.3}
        v = self.visual_encoder(visual_emb)
        t = self.text_encoder(text_emb)
        fused = self.fusion(torch.cat([v, t], dim=-1))
        predicted_ecr = self.ecr_head(fused).squeeze(-1)
        predicted_aesthetic = self.aesthetic_head(fused).squeeze(-1)
        predicted_technical = self.technical_head(fused).squeeze(-1)
        outputs = {'predicted_ecr': predicted_ecr, 'predicted_aesthetic': predicted_aesthetic,
                   'predicted_technical': predicted_technical, 'fused_hidden': fused}
        loss = torch.tensor(0.0, device=visual_emb.device)
        losses = {}
        if ecr_targets is not None:
            l = F.mse_loss(predicted_ecr, ecr_targets); losses['ecr_loss'] = l; loss = loss + loss_weights['ecr'] * l
        if aesthetic_targets is not None:
            l = F.mse_loss(predicted_aesthetic, aesthetic_targets); losses['aesthetic_loss'] = l; loss = loss + loss_weights['aesthetic'] * l
        if technical_targets is not None:
            l = F.mse_loss(predicted_technical, technical_targets); losses['technical_loss'] = l; loss = loss + loss_weights['technical'] * l
        if teacher_emb is not None:
            student_proj = self.kd_projector(fused)
            l = 1.0 - F.cosine_similarity(student_proj, teacher_emb, dim=-1).mean(); losses['kd_loss'] = l; loss = loss + loss_weights['kd'] * l
        outputs['loss'] = loss; outputs['losses'] = losses
        return outputs

print(f"ConcatStudent defined: {sum(p.numel() for p in ConcatStudent().parameters()):,} params")

In [ ]:
print("=" * 60)
print("ABLATION 2: Concat vs Gated Fusion")
print("=" * 60)

abl_train_ds = VideoFeaturesDataset(TRAIN_SPLIT)
abl_train_loader = DataLoader(abl_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
abl_val_ds = VideoFeaturesDataset(VAL_SPLIT)
abl_val_loader = DataLoader(abl_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

concat_model = ConcatStudent(hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
concat_model, concat_hist = train_ablation_model(
    concat_model, abl_train_loader, abl_val_loader, DEVICE,
    loss_weights=LOSS_WEIGHTS, epochs=EPOCHS, lr=LEARNING_RATE, name="Concat")

fusion_results = {}
fusion_results['Gated Fusion'] = evaluate_full_metrics(best_model, VAL_SPLIT, DEVICE, 'full')
fusion_results['Concatenation'] = evaluate_full_metrics(concat_model, VAL_SPLIT, DEVICE, 'full')

print(f"\n{'='*60}")
print(f"{'Fusion':<20} {'Pearson':>8} {'Spearman':>9} {'Kendall':>8} {'MAE':>8}")
print(f"{'='*60}")
for name, m in fusion_results.items():
    print(f"{name:<20} {m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['kendall']:>8.4f} {m['mae']:>8.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['ecr_pearson'], label='Gated', color='blue')
axes[0].plot(concat_hist['ecr_pearson'], label='Concat', color='orange')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('ECR Pearson'); axes[0].set_title('Gated vs Concat'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history['val_loss'], label='Gated', color='blue')
axes[1].plot(concat_hist['val_loss'], label='Concat', color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss'); axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ablation_fusion.png'), dpi=150, bbox_inches='tight')
plt.show()
del concat_model; gc.collect()

### 13.3 Ablation: Training Strategy

Compare: ECR-only, Multi-task (no KD), Full KD (proposed).

In [ ]:
print("=" * 60)
print("ABLATION 3: Training Strategy")
print("=" * 60)

strategies = {
    'ECR only':   {'ecr': 1.0, 'aesthetic': 0.0, 'technical': 0.0, 'kd': 0.0},
    'Multi-task': {'ecr': 1.0, 'aesthetic': 0.3, 'technical': 0.3, 'kd': 0.0},
    'Full KD':    {'ecr': 1.0, 'aesthetic': 0.3, 'technical': 0.3, 'kd': 0.3},
}

strategy_results = {}
strategy_histories = {}

abl_train_ds = VideoFeaturesDataset(TRAIN_SPLIT)
abl_train_loader = DataLoader(abl_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
abl_val_ds = VideoFeaturesDataset(VAL_SPLIT)
abl_val_loader = DataLoader(abl_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

for name, weights in strategies.items():
    print(f"\n--- Training: {name} ---")
    m = DistilStudent(hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
    m, h = train_ablation_model(m, abl_train_loader, abl_val_loader, DEVICE,
                                loss_weights=weights, epochs=EPOCHS, lr=LEARNING_RATE, name=name)
    metrics = evaluate_full_metrics(m, VAL_SPLIT, DEVICE, 'full')
    strategy_results[name] = metrics
    strategy_histories[name] = h
    print(f"  -> Pearson={metrics['pearson']:.4f}, Spearman={metrics['spearman']:.4f}, MAE={metrics['mae']:.4f}")
    del m; gc.collect()

print(f"\n{'='*70}")
print(f"{'Strategy':<15} {'Pearson':>8} {'Spearman':>9} {'Kendall':>8} {'MAE':>8} {'MSE':>10}")
print(f"{'='*70}")
for name, m in strategy_results.items():
    print(f"{name:<15} {m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['kendall']:>8.4f} {m['mae']:>8.4f} {m['mse']:>10.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'ECR only': '#DD8452', 'Multi-task': '#55A868', 'Full KD': '#4C72B0'}
for name, h in strategy_histories.items():
    axes[0].plot(h['ecr_pearson'], label=name, color=colors[name])
    axes[1].plot(h['val_loss'], label=name, color=colors[name])
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('ECR Pearson'); axes[0].set_title('Strategy Comparison'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss'); axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ablation_strategy.png'), dpi=150, bbox_inches='tight')
plt.show()

### 13.4 Comprehensive Results Summary

In [ ]:
print("=" * 80)
print("COMPREHENSIVE ABLATION RESULTS")
print("=" * 80)

all_results = [
    ('Visual Only (Gated)', modality_results['visual_only'], '2.8M'),
    ('Text Only (Gated)', modality_results['text_only'], '2.8M'),
    ('Concat (V+T)', fusion_results['Concatenation'], '2.6M'),
    ('Gated (V+T)', fusion_results['Gated Fusion'], '2.8M'),
]
for name, m in strategy_results.items():
    all_results.append((f'Gated + {name}', m, '2.8M'))

print(f"\n{'Model/Config':<25} {'Params':>7} {'Pearson':>8} {'Spearman':>9} {'Kendall':>8} {'MAE':>8} {'MSE':>10}")
print("-" * 80)
for name, m, params in all_results:
    print(f"{name:<25} {params:>7} {m['pearson']:>8.4f} {m['spearman']:>9.4f} {m['kendall']:>8.4f} {m['mae']:>8.4f} {m['mse']:>10.6f}")

best_p = max(all_results, key=lambda x: x[1]['pearson'])
print(f"\n\u2605 Best Pearson: {best_p[0]} ({best_p[1]['pearson']:.4f})")

ablation_summary = {'modality': modality_results, 'fusion': {k: v for k, v in fusion_results.items()},
                     'strategy': strategy_results, 'inference_ms': inference_ms,
                     'model_params': count_parameters(best_model)['total']}
with open(os.path.join(OUTPUT_DIR, 'ablation_results.json'), 'w') as f:
    json.dump(ablation_summary, f, indent=2)
print(f"\nSaved to {os.path.join(OUTPUT_DIR, 'ablation_results.json')}")

## 14. Summary & Conclusions

In [ ]:
print("=" * 60)
print("Distil-ShortVU: Video Engagement Prediction")
print("=" * 60)
print()
print("Architecture:")
print(f"  Model: DistilStudent (gated multimodal fusion)")
print(f"  Parameters: {count_parameters(best_model)['total']:,}")
print(f"  Visual teacher: ImageBind (1024-dim)")
print(f"  Text teacher: all-MiniLM-L6-v2 (384-dim)")
print(f"  Quality teacher: pyiqa MUSIQ + TOPIQ")
print()
print("Training:")
print(f"  Dataset: SnapUGC ({len(train_dataset)} train / {len(val_dataset)} val)")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Loss: L_ECR + {LOSS_WEIGHTS['aesthetic']}*L_aes + {LOSS_WEIGHTS['technical']}*L_tech + {LOSS_WEIGHTS['kd']}*L_KD")
print()
if has_ecr:
    from scipy.stats import spearmanr, kendalltau
    pearson = np.corrcoef(pred_ecr, true_ecr)[0, 1]
    spearman = spearmanr(pred_ecr, true_ecr).correlation
    kendall = kendalltau(pred_ecr, true_ecr).correlation
    mae = np.mean(np.abs(pred_ecr - true_ecr))
    mse = np.mean((pred_ecr - true_ecr)**2)
    binary_acc = np.mean((true_ecr > 0.5).astype(int) == (pred_ecr > 0.5).astype(int))
    print("Results (Validation):")
    print(f"  ECR Pearson:     {pearson:.4f}")
    print(f"  ECR Spearman:    {spearman:.4f}")
    print(f"  ECR Kendall:     {kendall:.4f}")
    print(f"  ECR MAE:         {mae:.4f}")
    print(f"  ECR MSE:         {mse:.6f}")
    print(f"  Binary Accuracy: {binary_acc:.4f}")
    print(f"  Inference Time:  {inference_ms:.2f} ms/sample")
    print()
print("Key Findings:")
print(f"  Visual contribution: {vis_arr.mean()/(vis_arr.mean()+txt_arr.mean()+1e-8)*100:.1f}%")
print(f"  Text contribution:   {txt_arr.mean()/(vis_arr.mean()+txt_arr.mean()+1e-8)*100:.1f}%")
print()
print("Output files:")
print(f"  Best model:       {BEST_MODEL_PATH}")
print(f"  Predictions:      {os.path.join(OUTPUT_DIR, 'val_predictions.json')}")
print(f"  Ablation results: {os.path.join(OUTPUT_DIR, 'ablation_results.json')}")
print("=" * 60)